## 🟦 **DAY 1 — 2 HRS**

### 1. **GENAI — 60 MIN**
- [ ] Transformers basics
- [ ] Tokenization
- [ ] Attention concept
- [ ] Hugging Face workflow

### 2. **DSA — 45 MIN**
- [ ] Arrays basics
- [ ] 2 easy problems

### 3. **CODING / PROJECT — 15 MIN**
- [ ] Set up today's GenAI workspace
- [ ] GitHub progress commit

<br>
<br>

## **TRANSFORMERS BASICS**

The Transformer is a deep learning architecture introduced in the 2017 paper *"Attention Is All You Need"*. It revolutionized natural language processing (NLP) by abandoning sequential processing (like RNNs/LSTMs) in favor of parallel processing, relying entirely on **Attention Mechanisms**.

### High-Level Architecture (ASCII Diagram)

Below is a simplified diagram of a standard Encoder block (the foundation for models like BERT) and how data flows through it.

```text
       Input Sequence (e.g., "The cat sat")
                 |
                 v
      +----------------------+
      |   Token Embeddings   |  <-- Converts words to vectors
      +----------------------+
                 |   +   <--------- [ Positional Encoding ] (Adds word order)
                 v
+------------------------------------+
|          TRANSFORMER BLOCK         |
|                                    |
|   +----------------------------+   |
|   | Multi-Head Self-Attention  |   | <-- Figures out context & relationships
|   +----------------------------+   |     between all words simultaneously.
|                 |                  |
|   +----------------------------+   |
|   |      Add & Normalize       |   | <-- Stabilizes training (Residuals)
|   +----------------------------+   |
|                 |                  |
|   +----------------------------+   |
|   |   Feed-Forward Network     |   | <-- Applies non-linear transformations
|   +----------------------------+   |
|                 |                  |
|   +----------------------------+   |
|   |      Add & Normalize       |   |
|   +----------------------------+   |
+------------------------------------+
                 |
                 v
   Contextualized Output Embeddings
       (Ready for next layers or
        final Linear/Softmax layer)

<br>

## Sentence Workflow through a Transformer

Here is an end-to-end workflow of how a Transformer processes a specific sentence. We will use the sentence **"The cat sat"** to trace the journey from raw text to contextualized output.

```text
Sentence: "The cat sat"

======================================================================
STEP 1: TOKENIZATION
(Breaking the sentence into smaller pieces and mapping to dictionary IDs)

 Words:       "The"             "cat"             "sat"
               |                 |                 |
 Token IDs:  [ 1996 ]          [ 4937 ]          [ 2938 ]

======================================================================
STEP 2: EMBEDDING & POSITIONAL ENCODING
(Converting IDs into arrays of numbers, and adding position tags so
 the model knows word order)

 Vectors:   [0.12, -0.3...]   [0.88, 0.41...]   [-0.5, 0.12...]
                   +                 +                 +
 Position:      [Pos 1]           [Pos 2]           [Pos 3]
                   |                 |                 |
                   v                 v                 v
 Input Embed:    ( E1 )            ( E2 )            ( E3 )

======================================================================
STEP 3: MULTI-HEAD SELF-ATTENTION (Inside the Transformer Block)
(This is where the magic happens. Every word looks at every other word
 to gather context. Notice how they all blend together.)

                 ( E1 )            ( E2 )            ( E3 )
                    \                |                /
                     \               |               /
                      \              |              /
                       +---------------------------+
                       |    SELF - ATTENTION       |  <-- "cat" looks at "The" and "sat"
                       +---------------------------+
                      /              |              \
                     /               |               \
                    /                |                \
                   v                 v                 v
                 ( O1 )            ( O2 )            ( O3 )

======================================================================
STEP 4: CONTEXTUALIZED OUTPUT
(The final vectors are no longer just dictionary definitions; they now
 contain the full meaning of the word *within the context of this exact sentence*.)

 O1 = "The" (aware that it precedes a feline sitting)
 O2 = "cat" (aware that it is "The" specific cat, and it just "sat")
 O3 = "sat" (aware that the action was performed by "The cat")

<br>

## Full Transformer Workflow with a Sentence

Here is the complete Encoder-Decoder architecture at work, using the example of translating the sentence **"The cat"** into French (**"Le chat"**).

This diagram shows **Step 1** of generation, where the model outputs the first translated word.

```text
====================================================================================
           FULL TRANSFORMER WORKFLOW (Translating: "The cat" -> "Le chat")
====================================================================================

               [ ENCODER TOWER ]                             [ DECODER TOWER ]
        (Understands the English input)               (Generates the French output)

                                                              Output: "Le" (98% prob)
                                                                       ^
                                                                       |
                                                             +-------------------+
                                                             |  Linear + Softmax | <-- Picks the best word
                                                             | (Vocabulary Math) |     from French dictionary
                                                             +-------------------+
                                                                       ^
                                                                       |
                                                             +-------------------+
                                                             |   Feed-Forward    | <-- Refines the prediction
                                                             +-------------------+
                                                                       ^
                                   Contextualized Vectors              |
         +--------------------+     (The deep meaning of     +-------------------+
         |    Feed-Forward    |       "The cat")             |  Encoder-Decoder  | <-- Cross-references the English
         +--------------------+ ---------------------------> |     Attention     |     context to find the French
                   ^                                         +-------------------+     equivalent.
                   |                                                   ^
         +--------------------+                                        |
         |  Self-Attention    | <-- "cat" looks at "The"     +-------------------+
         | (Multi-Head)       |     to understand context    | Masked Attention  | <-- Looks ONLY at previously
         +--------------------+                              |                   |     generated words (no cheating)
                   ^                                         +-------------------+
                   |                                                   ^
         +--------------------+                                        |
         |  Position Encoding | <-- Tags word order (1, 2)   +-------------------+
         +--------------------+                              | Position Encoding | <-- Tags word order
                   ^                                         +-------------------+
                   |                                                   ^
         +--------------------+                                        |
         |  Token Embeddings  | <-- Converts to vectors      +-------------------+
         +--------------------+                              |  Token Embeddings | <-- Converts to vectors
                   ^                                         +-------------------+
                   |                                                   ^
                   |                                                   |
         INPUT: ["The", "cat"]                               INPUT: [`<start>`]
                                                              (Step 1 of Generation)



---





---



<br>
<br>

## **HUGGING FACE WORKFLOW**

The Hugging Face `pipeline` is a high-level API that abstracts away the complexity of using [Transformers](https://huggingface.co/docs/transformers/en/index). Under the hood, whether you use a pre-built pipeline or write the code manually, every machine learning task follows the same fundamental three-step workflow.

### 1. **Tokenization (Pre-processing)**
Raw text cannot be fed directly into a mathematical model. The **Tokenizer** splits the input text into smaller chunks (words or sub-words called tokens), maps them to integer IDs based on a pre-trained vocabulary, and generates the necessary input tensors (like `input_ids` and `attention_mask`).

### 2. **Model Inference**
The core **Model** (e.g., BERT, GPT, Llama) takes the numerical tensors generated by the tokenizer and passes them through its neural network layers. It outputs raw, unnormalized mathematical scores known as **logits** or hidden states.

### 3. **Post-Processing**
The raw logits are not human-readable. **Post-processing** applies functions (like Softmax) to convert these raw scores into probabilities, and maps the highest probabilities back to human-understandable labels (like "Positive" or "Negative") or generates the final decoded text string.

### **Standard Workflow (ASCII Diagram)**

Below is the visual representation of how data flows from your raw text to the final prediction.

```text
======================================================================
               HUGGING FACE STANDARD WORKFLOW
======================================================================

                       [ Raw Input Text ]
                  (e.g., "I love this library!")
                               |
                               v
+--------------------------------------------------------------------+
| 1. TOKENIZER (Pre-processing)                                      |
|                                                                    |
|    - Splits text into tokens: ["I", "love", "this", "lib", "##rary"]
|    - Maps to IDs: [1045, 2293, 2023, ... ]                         |
|    - Adds Special Tokens: [CLS], [SEP]                             |
+--------------------------------------------------------------------+
                               |
                   Dictionary IDs & Tensors
                  (input_ids, attention_mask)
                               |
                               v
+--------------------------------------------------------------------+
| 2. TRANSFORMER MODEL (Inference)                                   |
|                                                                    |
|    - Feeds tensors into the neural network (Encoder/Decoder)       |
|    - Computes Attention                                            |
|    - Outputs raw mathematical predictions (Logits)                 |
+--------------------------------------------------------------------+
                               |
                         Raw Logits
                 (e.g., [-4.3, 5.1, -0.2])
                               |
                               v
+--------------------------------------------------------------------+
| 3. POST-PROCESSING                                                 |
|                                                                    |
|    - Applies Activation Functions (e.g., Softmax)                  |
|    - Converts math to probabilities (e.g., 98% Positive)           |
|    - Maps to human-readable labels                                 |
+--------------------------------------------------------------------+
                               |
                               v
                       [ Final Output ]
                (e.g., label: "POSITIVE", score: 0.98)

In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Choose a pre-trained model checkpoint
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"

# ==========================================
# 1. TOKENIZATION (Pre-Processing)
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

raw_text = "I love learning about Transformers and Generative AI!"
inputs = tokenizer(raw_text, return_tensors="pt")

print("--- 1. Tokenizer Output (Tensors) ---")
print("Input IDs:", inputs["input_ids"])
print("Tokens:", tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))
print("Attention Mask:", inputs["attention_mask"])

# ==========================================
# 2. MODEL INFERENCE (Raw Logits)
# ==========================================
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
print("\n--- 2. Model Output (Raw Logits) ---")
print("Logits:", logits)

# ==========================================
# 3. POST-PROCESSING (Probabilities & Labels)
# ==========================================
# Convert raw logits to probabilities using Softmax
probabilities = F.softmax(logits, dim=-1)[0]

# Map predictions to human-readable labels
predicted_class_id = torch.argmax(probabilities).item()
predicted_label = model.config.id2label[predicted_class_id]
confidence = probabilities[predicted_class_id].item()

print("\n--- 3. Post-Processed Prediction ---")
print(f"Label: {predicted_label} (Confidence: {confidence * 100:.2f}%)")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

--- 1. Tokenizer Output (Tensors) ---
Input IDs: tensor([[  101,  1045,  2293,  4083,  2055, 19081,  1998, 11416,  6024,  9932,
           999,   102]])
Tokens: ['[CLS]', 'i', 'love', 'learning', 'about', 'transformers', 'and', 'genera', '##tive', 'ai', '!', '[SEP]']
Attention Mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


--- 2. Model Output (Raw Logits) ---
Logits: tensor([[-3.4909,  3.7002]])

--- 3. Post-Processed Prediction ---
Label: POSITIVE (Confidence: 99.92%)




---





---



<br>

#### **PRATICE**

In [6]:
from transformers import AutoTokenizer , AutoModelForSequenceClassification
import torch.nn.functional as F
import torch

#MODEL
model_pkt = "distilbert-base-uncased-finetuned-sst-2-english"

#TOKENIZER
tkn = AutoTokenizer.from_pretrained(model_pkt)

#INITIALIZE THE MODEL
model = AutoModelForSequenceClassification.from_pretrained(model_pkt)

#IMPORT DATA
raw_text = ["Transformers are absolutely amazing!", "I am completely stuck and confused."]

#RESULT
res = tkn(raw_text, padding=True, return_tensors='pt')

#PRINT INPUT_IDS, ATTENTION_MASK
print("\n HERE INPUT_IDS< ATTENTION_MASK")
print("\n INPUT_IDS", res['input_ids'][0])
print("\n ATTENTION_MASK", res['attention_mask'][0])

#HERE MODEL INFERENCE LOGITS
with torch.no_grad():
    logits = model(**res).logits

#PRINT LOGITS
print("\n HERE LOGITS")
print("\n LOGITS", logits)


#BATCH POST PROCESSING
probabilities = F.softmax(logits, dim=-1)
# Get predicted class IDs for each sample in the batch
predicted_class_ids = torch.argmax(probabilities, dim=1)

#PRINT POST PROCESSING
print("\n HERE POST PROCESSING")
for i, pred_id in enumerate(predicted_class_ids):
    predicted_label = model.config.id2label[pred_id.item()]
    confidence = probabilities[i, pred_id].item() # Access the specific probability for the predicted class
    print(f"Sentence {i+1}: Label: {predicted_label} (Confidence: {confidence * 100:.2f}%)")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


 HERE INPUT_IDS< ATTENTION_MASK

 INPUT_IDS tensor([  101, 19081,  2024,  7078,  6429,   999,   102,     0,     0])

 ATTENTION_MASK tensor([1, 1, 1, 1, 1, 1, 1, 0, 0])

 HERE LOGITS

 LOGITS tensor([[-4.3364,  4.6614],
        [ 4.1286, -3.2767]])

 HERE POST PROCESSING
Sentence 1: Label: POSITIVE (Confidence: 99.99%)
Sentence 2: Label: NEGATIVE (Confidence: 99.94%)
